# Submission 02 - XGBoost

This submission uses the XGBoost model from the previous experiment.

The model achieved a validation ROC-AUC of 0.941635, which is higher than the previous HistGradientBoosting result of 0.940597.

For the competition submission, the model is trained on the full training dataset and predicts the probability of `Will_Buy_EV = Yes`.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

X_train = train.drop(columns=['Will_Buy_EV', 'id'])
y_train = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})
X_test = test.drop(columns=['id'])

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [ ]:
pipeline.fit(X_train, y_train)

test_predictions = pipeline.predict_proba(X_test)[:, 1]

submission = sample_submission.copy()
submission['Will_Buy_EV'] = test_predictions

output_path = '../submissions/submission_02.csv'
submission.to_csv(output_path, index=False)

print('Submission saved:', output_path)
print()
print('First 5 predictions:')
print(submission.head())

In [ ]:
assert submission.shape == sample_submission.shape
assert list(submission.columns) == list(sample_submission.columns)
assert submission['Will_Buy_EV'].notna().all()
assert submission['Will_Buy_EV'].between(0, 1).all()

print('Submission verification passed.')